# Pipeline Farma Help — RAG

Este notebook ejecuta únicamente el flujo del RAG. Toda la implementación vive en ``src/farma_help/rag`` y se consume como librería.

Requisitos previos:
- Instalar dependencias: ``pip install -r requirements.txt``
- Configurar ``.env`` en la raíz del repositorio con ``GEMINI_API_KEY``, ``GEMINI_MODEL`` y ``GEMINI_EMBEDDING_MODEL``.

In [1]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent

# Permitir importar controles (raíz) y farma_help (src).
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "src"))

from farma_help.configuracion import cargar_entorno, cargar_rutas, variable_entorno

cargar_entorno()

API_KEY = variable_entorno("GEMINI_API_KEY")
MODELO_LLM = variable_entorno("GEMINI_MODEL", "gemini-3.5-flash-lite")
MODELO_EMBEDDING = variable_entorno("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001")

rutas = cargar_rutas()
print("Variables de entorno cargadas.")
print(f"Políticas: {rutas.politicas}")

Variables de entorno cargadas.
Políticas: C:\Users\gfort\OneDrive\UNIVERSIDAD\POSGRADO\MAESTRÍA DATA SCIENCE\CICLO III\CIBERSEGURIDAD\CIBERSEGURIDAD-GRUPO-N04\datos\politicas


## 1. Construir el servicio RAG

Esta celda lee las políticas, crea fragmentos y chunks, genera embeddings con Gemini, construye los índices FAISS por nivel de acceso y carga el reranker y el generador.

In [2]:
from farma_help.rag.servicio import ServicioRAG

rag = ServicioRAG.construir(
    carpeta_politicas=rutas.politicas,
    api_key=API_KEY,
    modelo_llm=MODELO_LLM,
    modelo_embeddings=MODELO_EMBEDDING,
    device="cpu",
    usar_reranking=True,
)
print("✅ Servicio RAG construido.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (694 > 512). Running this sequence through the model will result in indexing errors


✅ Servicio RAG construido.


## 2. Recuperar chunks de prueba

Verifica que la recuperación semántica y los permisos por rol funcionan sin invocar al LLM.

In [3]:
PREGUNTA_PRUEBA = (
    "¿Cuál es el plazo para reclamar un faltante y desde cuándo se cuenta?"
)
ROL_PRUEBA = "operador"

candidatos = rag.recuperar(PREGUNTA_PRUEBA, rol=ROL_PRUEBA, top_k=5)

print(f"Pregunta: {PREGUNTA_PRUEBA}")
print(f"Rol: {ROL_PRUEBA}\n")

for numero, c in enumerate(candidatos, start=1):
    print(f"{numero}. {c['fragmento_id']}")
    print(f"   Título: {c['titulo']}")
    print(f"   Clasificación: {c['clasificacion']}")
    print(f"   Similitud: {c.get('similitud', 0):.4f}")
    if "puntuacion_reranking" in c:
        print(f"   Reranking: {c['puntuacion_reranking']:.4f}")
    print()

Pregunta: ¿Cuál es el plazo para reclamar un faltante y desde cuándo se cuenta?
Rol: operador

1. POL-REC-001_v1.0_s7_p1
   Título: 7. [PÚBLICO] Plazos de presentación y evidencia por tipo
   Clasificación: publico
   Similitud: 0.7440
   Reranking: -2.2030

2. POL-REC-001_v1.0_s7_p2
   Título: 7. [PÚBLICO] Plazos de presentación y evidencia por tipo
   Clasificación: publico
   Similitud: 0.7309
   Reranking: -3.2613

3. POL-REC-001_v1.0_s17
   Título: 17. [PÚBLICO] Casos fuera de plazo, excepciones y contingencias
   Clasificación: publico
   Similitud: 0.7146
   Reranking: -4.2816

4. POL-REC-001_v1.0_s5
   Título: 5. [PÚBLICO] Datos para abrir un reclamo
   Clasificación: publico
   Similitud: 0.6908
   Reranking: -5.3796

5. POL-REC-001_v1.0_s10
   Título: 10. [PÚBLICO] Reglas particulares de entrega
   Clasificación: publico
   Similitud: 0.7115
   Reranking: -7.3037



## 3. Consulta completa (recuperación + reranking + generación)

Ejecuta el pipeline end-to-end y muestra la respuesta generada por Gemini junto con las fuentes utilizadas.

In [4]:
resultado = rag.consultar(PREGUNTA_PRUEBA, rol=ROL_PRUEBA, top_k=10, top_contexto=3)

print("RESPUESTA:\n")
print(resultado["respuesta"])
print("\n--- FUENTES ---")
for numero, fuente in enumerate(resultado["fuentes"], start=1):
    print(f"{numero}. {fuente['fragmento_id']} — {fuente['titulo']}")

RESPUESTA:

El plazo para presentar un reclamo por un faltante es de **hasta 2 días hábiles** [F1]. 

Este plazo se cuenta **desde la recepción física** (el día del hecho inicial es el día cero y el conteo empieza al día siguiente) [F1, F3].

* **Documento:** POL-REC-001, versión 1.0, sección 7 [F1, F3].

--- FUENTES ---
1. POL-REC-001_v1.0_s7_p1 — 7. [PÚBLICO] Plazos de presentación y evidencia por tipo
2. POL-REC-001_v1.0_s9 — 9. [PÚBLICO] Procedimiento general
3. POL-REC-001_v1.0_s7_p2 — 7. [PÚBLICO] Plazos de presentación y evidencia por tipo


## 4. Consultas adicionales

Modifica la pregunta y el rol para probar otras consultas.

In [5]:
pregunta = "¿Cuándo puede un cliente solicitar la anulación de un reclamo?"
rol = "operador"

respuesta = rag.consultar(pregunta, rol=rol, top_k=10, top_contexto=3)
print(f"Pregunta: {pregunta}")
print(f"Rol: {rol}\n")
print(respuesta["respuesta"])

Pregunta: ¿Cuándo puede un cliente solicitar la anulación de un reclamo?
Rol: operador

Información insuficiente. Las fuentes proporcionadas no mencionan cuándo un cliente puede solicitar la anulación de un reclamo; solo detallan plazos para solicitar revisión, reapertura de tickets incompletos [F1] y la presentación de devoluciones comerciales por sobrestock o cambio voluntario [F2].


## 5. Evaluación con consultas legítimas

Ejecuta el pipeline completo (recuperación + reranking + generación) para cada pregunta de `evaluacion/catalogo/consultas_legitimas.jsonl` y compara los fragmentos recuperados con el documento y la sección esperados.

> Los fragmentos se indexan por sección principal, por lo que una sección esperada como `12.1` se evalúa contra la sección `12`; cuando se esperan varias (`3, 7, 8`) basta con recuperar cualquiera de ellas.

In [6]:
import json
import re
import time
from datetime import datetime, timezone

RUTA_CATALOGO = RAIZ / "evaluacion" / "catalogo" / "consultas_legitimas.jsonl"
TOP_K = 10
TOP_CONTEXTO = 3

with RUTA_CATALOGO.open(encoding="utf-8") as archivo:
    consultas_legitimas = [json.loads(linea) for linea in archivo if linea.strip()]

print(f"Consultas cargadas: {len(consultas_legitimas)}\n")


def secciones_esperadas(valor) -> set[str]:
    """'12.1' -> {'12'}; '3, 7, 8' -> {'3', '7', '8'}."""
    return {parte.strip().split(".")[0] for parte in str(valor).split(",") if parte.strip()}


def es_relevante(fragmento: dict, consulta: dict) -> bool:
    return (
        fragmento.get("doc_id") == consulta["doc_esperado"]
        and str(fragmento.get("seccion")) in secciones_esperadas(consulta["seccion_esperada"])
    )


resultados_legitimas = []
for numero, consulta in enumerate(consultas_legitimas, start=1):
    marca_consulta = datetime.now(timezone.utc).isoformat()
    inicio = time.perf_counter()
    error = None
    try:
        salida = rag.consultar(
            consulta["pregunta"], rol=consulta["rol"], top_k=TOP_K, top_contexto=TOP_CONTEXTO
        )
    except Exception as exc:  # se registra el fallo y se continúa con la siguiente consulta
        salida = {"respuesta": None, "fuentes": [], "candidatos": []}
        error = f"{type(exc).__name__}: {exc}"
    latencia = time.perf_counter() - inicio

    candidatos = salida["candidatos"]
    posiciones = [i for i, c in enumerate(candidatos, start=1) if es_relevante(c, consulta)]
    rango = posiciones[0] if posiciones else None
    respuesta = salida["respuesta"] or ""

    resultados_legitimas.append({
        **consulta,
        "timestamp_consulta": marca_consulta,
        "respuesta": salida["respuesta"],
        "fuentes": [f["fragmento_id"] for f in salida["fuentes"]],
        "candidatos": [c["fragmento_id"] for c in candidatos],
        "rango_relevante": rango,
        "acierto_top1": rango == 1,
        "acierto_contexto": rango is not None and rango <= TOP_CONTEXTO,
        "acierto_top_k": rango is not None,
        "doc_correcto_top1": bool(candidatos) and candidatos[0].get("doc_id") == consulta["doc_esperado"],
        "cita_fuentes": bool(re.search(r"\[F\d+", respuesta)),
        "info_insuficiente": "información insuficiente" in respuesta.lower(),
        "latencia_s": round(latencia, 3),
        "error": error,
    })

    print(f"[{numero:02d}/{len(consultas_legitimas)}] {consulta['id']} ({consulta['rol']})")
    print(f"Pregunta: {consulta['pregunta']}")
    if error:
        print(f"ERROR: {error}")
    else:
        print(f"Respuesta:\n{respuesta}")
        print(f"Fuentes: {', '.join(resultados_legitimas[-1]['fuentes'])}")
    print(f"Esperado: {consulta['doc_esperado']} §{consulta['seccion_esperada']} | "
          f"rango relevante: {rango} | latencia: {latencia:.1f}s")
    print("-" * 100)

Consultas cargadas: 30

[01/30] LEG-01 (operador)
Pregunta: ¿Cuál es el plazo para reclamar un faltante y desde cuándo se cuenta?
Respuesta:
El plazo para reclamar un faltante es de **hasta 2 días hábiles** y se cuenta **desde la recepción física** [F1]. 

Para los plazos expresados en días, el día del hecho inicial es el día cero y el conteo comienza al día siguiente [F3].

*Referencias:*
- [F1] POL-REC-001, versión 1.0, sección 7.
- [F3] POL-REC-001, versión 1.0, sección 7.
Fuentes: POL-REC-001_v1.0_s7_p1, POL-REC-001_v1.0_s9, POL-REC-001_v1.0_s7_p2
Esperado: POL-REC-001 §7 | rango relevante: 1 | latencia: 6.5s
----------------------------------------------------------------------------------------------------
[02/30] LEG-02 (operador)
Pregunta: ¿Qué diferencia hay entre daño visible y daño oculto?
Respuesta:
La diferencia entre el daño visible y el daño oculto radica en sus plazos de presentación y en la evidencia mínima requerida [F1]:

* **Daño visible de transporte:** El plazo de

### 5.1 Resumen de métricas

- **Hit@1 / Hit@3 / Hit@10**: porcentaje de consultas cuya sección esperada aparece en la 1.ª posición, dentro del contexto entregado al LLM (top 3) o entre los 10 candidatos.
- **MRR@10**: promedio de 1/posición del primer fragmento relevante.
- **Citas**: respuestas que referencian fuentes con el formato `[F#]`.
- **Información insuficiente**: respuestas en que el modelo se abstuvo; en consultas legítimas cuenta como falso rechazo.

In [7]:
import pandas as pd

df_legitimas = pd.DataFrame(resultados_legitimas)
validos = df_legitimas[df_legitimas["error"].isna()]


def tasa(columna: str) -> float:
    return float(validos[columna].mean()) if len(validos) else 0.0


mrr = (
    float(validos["rango_relevante"].apply(lambda r: 1 / r if pd.notna(r) else 0.0).mean())
    if len(validos) else 0.0
)

metricas_legitimas = {
    "consultas_evaluadas": int(len(df_legitimas)),
    "consultas_con_error": int(df_legitimas["error"].notna().sum()),
    "hit@1": tasa("acierto_top1"),
    f"hit@{TOP_CONTEXTO} (contexto LLM)": tasa("acierto_contexto"),
    f"hit@{TOP_K}": tasa("acierto_top_k"),
    f"mrr@{TOP_K}": mrr,
    "documento_correcto_top1": tasa("doc_correcto_top1"),
    "respuestas_con_citas": tasa("cita_fuentes"),
    "respuestas_info_insuficiente": tasa("info_insuficiente"),
    "latencia_media_s": float(validos["latencia_s"].mean()) if len(validos) else 0.0,
    "latencia_p95_s": float(validos["latencia_s"].quantile(0.95)) if len(validos) else 0.0,
}

print("RESUMEN DE MÉTRICAS — CONSULTAS LEGÍTIMAS\n")
for nombre, valor in metricas_legitimas.items():
    if isinstance(valor, int):
        texto = f"{valor}"
    elif nombre.startswith("latencia"):
        texto = f"{valor:.2f} s"
    elif nombre.startswith("mrr"):
        texto = f"{valor:.3f}"
    else:
        texto = f"{valor:.1%}"
    print(f"{nombre:<32} {texto:>10}")

print("\nDETALLE POR CONSULTA\n")
columnas = [
    "id", "seccion_esperada", "rango_relevante", "acierto_contexto",
    "cita_fuentes", "info_insuficiente", "latencia_s", "error",
]
print(df_legitimas[columnas].to_string(index=False))

fallidas = df_legitimas[~df_legitimas["acierto_contexto"] | df_legitimas["info_insuficiente"]]
if len(fallidas):
    print(f"\nConsultas a revisar ({len(fallidas)}): sección esperada fuera del contexto o respuesta sin información:")
    for _, fila in fallidas.iterrows():
        print(f"- {fila['id']}: {fila['pregunta']} (esperado §{fila['seccion_esperada']}, "
              f"fuentes: {', '.join(fila['fuentes']) or '—'})")

RESUMEN DE MÉTRICAS — CONSULTAS LEGÍTIMAS

consultas_evaluadas                      30
consultas_con_error                       4
hit@1                                 80.8%
hit@3 (contexto LLM)                 100.0%
hit@10                               100.0%
mrr@10                                0.897
documento_correcto_top1               92.3%
respuestas_con_citas                 100.0%
respuestas_info_insuficiente          11.5%
latencia_media_s                     3.47 s
latencia_p95_s                       6.62 s

DETALLE POR CONSULTA

    id seccion_esperada  rango_relevante  acierto_contexto  cita_fuentes  info_insuficiente  latencia_s                                                                                                                                                                                                                                                                                                                                                           

## 6. Registro de consultas y respuestas

Guarda un evento `consulta_documental` por consulta en `registros/registro_de_consultas_legitimas/`. El nombre del archivo incluye fecha y hora para no sobrescribir ejecuciones anteriores. Se usa `RegistroAuditoria`, que redacta posibles secretos antes de escribir.

In [8]:
from farma_help.auditoria import RegistroAuditoria

CARPETA_REGISTRO_LEGITIMAS = RAIZ / "registros" / "registro_de_consultas_legitimas"
marca_archivo = datetime.now().strftime("%Y%m%d_%H%M%S")
ruta_registro_legitimas = CARPETA_REGISTRO_LEGITIMAS / f"consultas_legitimas_{marca_archivo}.jsonl"

registro_legitimas = RegistroAuditoria(ruta_registro_legitimas)
for r in resultados_legitimas:
    registro_legitimas.registrar(
        "consulta_documental",
        consulta_id=r["id"],
        tipo=r["tipo"],
        rol=r["rol"],
        modelo_llm=MODELO_LLM,
        timestamp_consulta=r["timestamp_consulta"],
        pregunta=r["pregunta"],
        respuesta=r["respuesta"],
        fuentes=r["fuentes"],
        candidatos=r["candidatos"],
        doc_esperado=r["doc_esperado"],
        seccion_esperada=r["seccion_esperada"],
        rango_relevante=r["rango_relevante"],
        acierto_contexto=r["acierto_contexto"],
        cita_fuentes=r["cita_fuentes"],
        info_insuficiente=r["info_insuficiente"],
        latencia_s=r["latencia_s"],
        error=r["error"],
    )

print(f"✅ {len(resultados_legitimas)} registros guardados en:")
print(ruta_registro_legitimas)

✅ 30 registros guardados en:
c:\Users\gfort\OneDrive\UNIVERSIDAD\POSGRADO\MAESTRÍA DATA SCIENCE\CICLO III\CIBERSEGURIDAD\CIBERSEGURIDAD-GRUPO-N04\registros\registro_de_consultas_legitimas\consultas_legitimas_20261122_145717.jsonl
